In [ ]:
from hcc_release_paths import project_path as _hcc_path
# Cell 1 — Step34 setup and frozen input inventory

from pathlib import Path
import pandas as pd
import numpy as np
import json
import sys
import os

print("=" * 90)
print("Cell 1 — Step34 setup and frozen input inventory")
print("=" * 90)

TCGA_ROOT = Path(_hcc_path(''))

STEP34 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step34_Virtual_State_Transition"
)

DATA = STEP34 / "Data"
FIGURES = STEP34 / "Figures"
LOGS = STEP34 / "Logs"
REFERENCE = STEP34 / "Reference"
RESULTS = STEP34 / "Results"

for p in [
    DATA,
    FIGURES,
    LOGS,
    REFERENCE,
    RESULTS
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# Upstream step folders
# ============================================================

STEP33 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step33_Virtual_Knockout"
)

STEP26 = (
    TCGA_ROOT /
    "LIHC_scRNA_Step26_MS3_Biology"
)


# ============================================================
# Helper: require exactly one source file
# ============================================================

def find_unique(root, pattern):

    hits = sorted(
        root.rglob(pattern)
    )

    print(
        f"{pattern}: {len(hits)} match(es)"
    )

    for x in hits:
        print("  -", x)

    assert len(hits) == 1, (
        f"Expected exactly one {pattern}, "
        f"found {len(hits)}"
    )

    return hits[0]


# ============================================================
# Frozen Step33 CellOracle
# ============================================================

ORACLE_PATH = find_unique(
    STEP33,
    "Step33_CellOracle_FORMAL_FITTED.celloracle.oracle"
)

STEP33_LOCK = find_unique(
    STEP33,
    "Step33_FINAL_LOCK.json"
)


# ============================================================
# Frozen Step26 biology/state object
# ============================================================

STEP26_H5AD = find_unique(
    STEP26,
    "GSE149614_STEP26_FINAL_FROZEN_MS3_BIOLOGY.h5ad"
)


# ============================================================
# Frozen Step27 trajectory object
# Search globally because folder naming is not assumed
# ============================================================

step27_hits = sorted(
    TCGA_ROOT.rglob(
        "GSE149614_STEP27_FINAL_TRAJECTORY.h5ad"
    )
)

print(
    "\nGSE149614_STEP27_FINAL_TRAJECTORY.h5ad:",
    len(step27_hits),
    "match(es)"
)

for x in step27_hits:
    print("  -", x)

assert len(step27_hits) == 1

STEP27_H5AD = step27_hits[0]


# ============================================================
# Source inventory
# ============================================================

sources = [
    {
        "role":
            "Frozen fitted CellOracle",

        "step":
            "Step33",

        "path":
            str(ORACLE_PATH),

        "size_GB":
            ORACLE_PATH.stat().st_size / 1024**3,
    },

    {
        "role":
            "Step33 final lock",

        "step":
            "Step33",

        "path":
            str(STEP33_LOCK),

        "size_GB":
            STEP33_LOCK.stat().st_size / 1024**3,
    },

    {
        "role":
            "Frozen malignant-state biology",

        "step":
            "Step26",

        "path":
            str(STEP26_H5AD),

        "size_GB":
            STEP26_H5AD.stat().st_size / 1024**3,
    },

    {
        "role":
            "Frozen trajectory/pseudotime",

        "step":
            "Step27",

        "path":
            str(STEP27_H5AD),

        "size_GB":
            STEP27_H5AD.stat().st_size / 1024**3,
    },
]

source_df = pd.DataFrame(sources)

source_df.to_csv(
    RESULTS /
    "Step34A_FrozenInputInventory.csv",
    index=False
)

reference_json = {
    "Step34_question":
        (
            "Counterfactual transcriptional-state "
            "displacement after virtual regulator suppression"
        ),

    "formal_TFs": [
        "FOXM1",
        "MYBL2",
        "E2F8",
        "TFDP1",
        "E2F1",
    ],

    "frozen_sources": sources,

    "upstream_objects_modified":
        False,
}

with open(
    REFERENCE /
    "Step34A_FrozenInputSources.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        reference_json,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nFrozen input inventory:")
print(source_df.to_string(index=False))

print()
print("Step34 root:")
print(STEP34)

print("=" * 90)
print("Cell 1 complete")
print("=" * 90)

In [ ]:
# Cell 2 — Load and audit frozen fitted CellOracle

print("=" * 90)
print("Cell 2 — Frozen CellOracle audit")
print("=" * 90)

import celloracle as co
import psutil

print("Python:", sys.version.split()[0])
print("CellOracle:", getattr(co, "__version__", "unknown"))

print(
    "Available RAM before load:",
    round(
        psutil.virtual_memory().available / 1024**3,
        2
    ),
    "GB"
)

assert hasattr(
    co,
    "load_hdf5"
), "celloracle.load_hdf5 not available"


try:

    oracle34 = co.load_hdf5(
        file_path=str(ORACLE_PATH)
    )

except TypeError:

    oracle34 = co.load_hdf5(
        str(ORACLE_PATH)
    )


adata34 = oracle34.adata

print(
    "\nLoaded Oracle shape:",
    adata34.shape
)

print(
    "Available RAM after load:",
    round(
        psutil.virtual_memory().available / 1024**3,
        2
    ),
    "GB"
)


# ============================================================
# Basic structure
# ============================================================

layer_keys = list(
    adata34.layers.keys()
)

obsm_keys = list(
    adata34.obsm.keys()
)

varm_keys = list(
    adata34.varm.keys()
)

uns_keys = list(
    adata34.uns.keys()
)

print("\nLayers:")
print(layer_keys)

print("\nobsm:")
print(obsm_keys)

print("\nvarm:")
print(varm_keys)

print("\nSelected uns keys:")
print(uns_keys[:50])


# ============================================================
# Frozen annotations
# ============================================================

required_obs = [
    "final_malignant_state_id",
    "patient",
]

for col in required_obs:

    assert col in adata34.obs.columns, (
        f"Missing obs column: {col}"
    )


state_counts = (
    adata34.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)

patient_counts = (
    adata34.obs[
        "patient"
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)

ms3_mask = (
    adata34.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
    ==
    "MS3"
)

ms3_patient_counts = (
    adata34.obs.loc[
        ms3_mask,
        "patient"
    ]
    .astype(str)
    .value_counts()
    .sort_index()
)


print("\nState counts:")
print(state_counts.to_string())

print("\nMS3 patient counts:")
print(ms3_patient_counts.to_string())

assert int(ms3_mask.sum()) == 720


# ============================================================
# Check fitted-GRN / simulation readiness
# ============================================================

audit = {
    "oracle_shape":
        list(adata34.shape),

    "layers":
        layer_keys,

    "obsm":
        obsm_keys,

    "varm":
        varm_keys,

    "has_imputed_count":
        "imputed_count" in layer_keys,

    "has_simulated_count_at_load":
        "simulated_count" in layer_keys,

    "has_frozen_harmony_rep":
        (
            "X_pca_harmony_lineage_clean"
            in obsm_keys
        ),

    "MS3_cells":
        int(ms3_mask.sum()),

    "MS3_patient_counts":
        {
            str(k): int(v)
            for k, v
            in ms3_patient_counts.items()
        },

    "all_regulatory_genes_n":
        len(
            getattr(
                oracle34,
                "all_regulatory_genes_in_TFdict",
                []
            )
        ),

    "active_regulatory_genes_n":
        len(
            getattr(
                oracle34,
                "active_regulatory_genes",
                []
            )
        ),
}

with open(
    LOGS /
    "Step34A_FrozenOracle_Audit.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nAudit summary:")
print(
    json.dumps(
        audit,
        indent=2,
        ensure_ascii=False
    )
)

print()
print("=" * 90)
print("Cell 2 complete")
print("=" * 90)

In [ ]:
# Cell 3 — Step26/27 frozen state-space alignment audit

print("=" * 90)
print("Cell 3 — Frozen state-space alignment audit")
print("=" * 90)

import anndata as ad

# Read only / backed mode
a26 = ad.read_h5ad(
    STEP26_H5AD,
    backed="r"
)

a27 = ad.read_h5ad(
    STEP27_H5AD,
    backed="r"
)


def inspect_adata(label, a):

    info = {
        "label":
            label,

        "shape":
            list(a.shape),

        "obs_columns":
            list(a.obs.columns),

        "obsm_keys":
            list(a.obsm.keys()),

        "varm_keys":
            list(a.varm.keys()),

        "uns_keys":
            list(a.uns.keys()),

        "layers":
            list(a.layers.keys()),

        "has_raw":
            a.raw is not None,
    }

    return info


info26 = inspect_adata(
    "Step26",
    a26
)

info27 = inspect_adata(
    "Step27",
    a27
)


# ============================================================
# Cell identity/order alignment
# ============================================================

oracle_cells = np.asarray(
    adata34.obs_names.astype(str)
)

cells26 = np.asarray(
    a26.obs_names.astype(str)
)

cells27 = np.asarray(
    a27.obs_names.astype(str)
)

same_order_26 = np.array_equal(
    oracle_cells,
    cells26
)

same_order_27 = np.array_equal(
    oracle_cells,
    cells27
)

same_set_26 = (
    set(oracle_cells)
    ==
    set(cells26)
)

same_set_27 = (
    set(oracle_cells)
    ==
    set(cells27)
)


print("\nCell alignment:")
print(
    "Oracle vs Step26 same set:",
    same_set_26
)
print(
    "Oracle vs Step26 same order:",
    same_order_26
)

print(
    "Oracle vs Step27 same set:",
    same_set_27
)
print(
    "Oracle vs Step27 same order:",
    same_order_27
)

assert same_set_26
assert same_set_27


# ============================================================
# Frozen state-label consistency
# ============================================================

state26 = (
    a26.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
)

state27 = (
    a27.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
)

state_oracle = (
    adata34.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
)

state26 = state26.reindex(
    adata34.obs_names
)

state27 = state27.reindex(
    adata34.obs_names
)

state26_match = bool(
    (
        state26.values
        ==
        state_oracle.values
    ).all()
)

state27_match = bool(
    (
        state27.values
        ==
        state_oracle.values
    ).all()
)

print()
print(
    "Step26 state labels identical:",
    state26_match
)

print(
    "Step27 state labels identical:",
    state27_match
)

assert state26_match
assert state27_match


# ============================================================
# State-space objects
# ============================================================

print("\nSTEP26 obsm:")
print(info26["obsm_keys"])

print("\nSTEP26 varm:")
print(info26["varm_keys"])

print("\nSTEP26 PCA/Harmony-related uns keys:")
print([
    x
    for x in info26["uns_keys"]
    if (
        "pca" in x.lower()
        or
        "harmony" in x.lower()
        or
        "neighbor" in x.lower()
    )
])

print("\nSTEP27 obsm:")
print(info27["obsm_keys"])

print("\nSTEP27 varm:")
print(info27["varm_keys"])

print("\nSTEP27 PCA/Harmony-related uns keys:")
print([
    x
    for x in info27["uns_keys"]
    if (
        "pca" in x.lower()
        or
        "harmony" in x.lower()
        or
        "neighbor" in x.lower()
        or
        "paga" in x.lower()
    )
])


# ============================================================
# Explicit projection-feasibility flags
# ============================================================

audit_state_space = {
    "oracle_vs_step26_same_cell_set":
        same_set_26,

    "oracle_vs_step26_same_order":
        same_order_26,

    "oracle_vs_step27_same_cell_set":
        same_set_27,

    "oracle_vs_step27_same_order":
        same_order_27,

    "state_labels_step26_identical":
        state26_match,

    "state_labels_step27_identical":
        state27_match,

    "Step26_has_X_pca_harmony_lineage_clean":
        (
            "X_pca_harmony_lineage_clean"
            in info26["obsm_keys"]
        ),

    "Step27_has_X_pca_harmony_lineage_clean":
        (
            "X_pca_harmony_lineage_clean"
            in info27["obsm_keys"]
        ),

    "Step26_has_PCs_loadings":
        "PCs" in info26["varm_keys"],

    "Step27_has_PCs_loadings":
        "PCs" in info27["varm_keys"],

    "Step26_has_pca_uns":
        "pca" in info26["uns_keys"],

    "Step27_has_pca_uns":
        "pca" in info27["uns_keys"],
}

with open(
    LOGS /
    "Step34A_FrozenStateSpace_Audit.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        audit_state_space,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("Projection-feasibility flags:")
print(
    json.dumps(
        audit_state_space,
        indent=2,
        ensure_ascii=False
    )
)

print()
print("=" * 90)
print("Cell 3 complete")
print("=" * 90)

In [ ]:
# Cell 4 — Step34A REPORT

print("=" * 90)
print("STEP34A — FROZEN INPUT / STATE-SPACE REPORT")
print("=" * 90)

print("\nFrozen CellOracle")
print(
    " Shape:",
    adata34.shape
)

print(
    " imputed_count:",
    "imputed_count"
    in adata34.layers
)

print(
    " simulated_count already present:",
    "simulated_count"
    in adata34.layers
)

print(
    " frozen Harmony coordinates in Oracle:",
    "X_pca_harmony_lineage_clean"
    in adata34.obsm
)


print("\nMS3")
print(
    " cells:",
    int(ms3_mask.sum())
)

print(
    " patients:"
)

print(
    ms3_patient_counts.to_string()
)


print("\nCross-step alignment")

print(
    " Oracle ↔ Step26 same cells:",
    same_set_26
)

print(
    " Oracle ↔ Step27 same cells:",
    same_set_27
)

print(
    " Step26 state labels identical:",
    state26_match
)

print(
    " Step27 state labels identical:",
    state27_match
)


print("\nFrozen state-space projection ingredients")

print(
    " Step26 Harmony coordinates:",
    audit_state_space[
        "Step26_has_X_pca_harmony_lineage_clean"
    ]
)

print(
    " Step26 PCA loadings:",
    audit_state_space[
        "Step26_has_PCs_loadings"
    ]
)

print(
    " Step26 PCA metadata:",
    audit_state_space[
        "Step26_has_pca_uns"
    ]
)

print(
    " Step27 Harmony coordinates:",
    audit_state_space[
        "Step27_has_X_pca_harmony_lineage_clean"
    ]
)

print(
    " Step27 PCA loadings:",
    audit_state_space[
        "Step27_has_PCs_loadings"
    ]
)

print(
    " Step27 PCA metadata:",
    audit_state_space[
        "Step27_has_pca_uns"
    ]
)


report = {
    "status":
        "STEP34A_INPUT_AUDIT_COMPLETE",

    "MS3_cells":
        int(ms3_mask.sum()),

    "primary_patients":
        [
            "HCC08",
            "HCC09",
            "HCC10"
        ],

    "cell_alignment_pass":
        bool(
            same_set_26
            and
            same_set_27
        ),

    "state_annotation_alignment_pass":
        bool(
            state26_match
            and
            state27_match
        ),

    "state_space_audit":
        audit_state_space,

    "next_decision":
        (
            "Select a frozen projection strategy "
            "for counterfactual state displacement "
            "without state redefinition."
        ),

    "claim_boundary":
        (
            "Counterfactual transcriptional-state "
            "displacement only; not lineage conversion."
        )
}

REPORT_PATH = (
    LOGS /
    "Step34A_FINAL_REPORT.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )


print("\nSaved:")
print(REPORT_PATH)

print()
print("=" * 90)
print("STEP34A STATUS: COMPLETE")
print("=" * 90)

## Step34C. Native CellOracle State-Transition Analysis

The surrogate expression-to-Harmony encoder failed patient-grouped out-of-sample validation
and is therefore excluded from formal Step34 inference.

Formal Step34 analysis will use CellOracle's native perturbation transition-probability framework.

Primary question:

Do virtual suppressions of FOXM1, MYBL2, E2F8, TFDP1, or E2F1 alter the predicted
transition propensity of frozen MS3 malignant cells toward MS5 and/or MS6 states?

Frozen MS1–MS8 labels are used only as predefined destination states.

No state redefinition, reclustering, GRN refitting, or lineage-conversion claim is permitted.

In [ ]:
# Cell 11 — Native CellOracle transition API audit

print("=" * 90)
print("Cell 11 — Native CellOracle transition API audit")
print("=" * 90)

import inspect

print("Oracle embedding_name:")
print(
    getattr(
        oracle34,
        "embedding_name",
        None
    )
)

print("\nOracle embedding shape:")
embedding34 = getattr(
    oracle34,
    "embedding",
    None
)

if embedding34 is None:
    print("None")
else:
    print(
        np.asarray(
            embedding34
        ).shape
    )


print("\nAvailable obsm keys:")
for key in adata34.obsm.keys():

    arr = adata34.obsm[key]

    try:
        shape = arr.shape
    except Exception:
        shape = "unknown"

    print(
        f" - {key}: {shape}"
    )


print("\nestimate_transition_prob signature:")
print(
    inspect.signature(
        oracle34.estimate_transition_prob
    )
)

print("\ncalculate_embedding_shift signature:")
print(
    inspect.signature(
        oracle34.calculate_embedding_shift
    )
)


# ============================================================
# Selected source audit
# ============================================================

def selected_source_lines(func, keywords):

    src = inspect.getsource(
        func
    )

    lines = src.splitlines()

    selected = []

    for i, line in enumerate(
        lines,
        start=1
    ):

        if any(
            k.lower()
            in line.lower()
            for k in keywords
        ):

            selected.append(
                f"{i:03d}: {line}"
            )

    return selected


print(
    "\nSelected estimate_transition_prob source lines:"
)

for line in selected_source_lines(
    oracle34.estimate_transition_prob,
    [
        "simulated",
        "imputed",
        "embedding",
        "corr",
        "transition",
        "knn",
        "random"
    ]
):
    print(line)


print(
    "\nSelected calculate_embedding_shift source lines:"
)

for line in selected_source_lines(
    oracle34.calculate_embedding_shift,
    [
        "embedding",
        "transition",
        "delta",
        "random"
    ]
):
    print(line)


api_audit = {
    "embedding_name":
        str(
            getattr(
                oracle34,
                "embedding_name",
                None
            )
        ),

    "embedding_shape":
        (
            list(
                np.asarray(
                    embedding34
                ).shape
            )
            if embedding34 is not None
            else None
        ),

    "obsm": {
        key:
            list(
                adata34.obsm[key].shape
            )

        for key
        in adata34.obsm.keys()
    },

    "estimate_transition_prob_signature":
        str(
            inspect.signature(
                oracle34.estimate_transition_prob
            )
        ),

    "calculate_embedding_shift_signature":
        str(
            inspect.signature(
                oracle34.calculate_embedding_shift
            )
        )
}

with open(
    LOGS /
    "Step34C_CellOracleTransition_API_Audit.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        api_audit,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("=" * 90)
print("Cell 11 complete")
print("=" * 90)

In [ ]:
# Cell 12 — Frozen local destination availability audit

print("=" * 90)
print("Cell 12 — Frozen destination availability")
print("=" * 90)

from sklearn.neighbors import NearestNeighbors

HARMONY_KEY = (
    "X_pca_harmony_lineage_clean"
)

H = np.asarray(
    adata34.obsm[
        HARMONY_KEY
    ],
    dtype=np.float32
)

assert H.shape == (
    12057,
    30
)

N_NEIGHBORS_TRANSITION = 200

nn34 = NearestNeighbors(
    n_neighbors=(
        N_NEIGHBORS_TRANSITION
        + 1
    ),
    metric="euclidean",
    n_jobs=4
)

nn34.fit(H)

_, nbr_idx = nn34.kneighbors(
    H
)

# remove self
nbr_idx = nbr_idx[:, 1:]


state_array = (
    adata34.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
    .values
)

patient_array = (
    adata34.obs[
        "patient"
    ]
    .astype(str)
    .values
)

ms3_idx = np.where(
    state_array == "MS3"
)[0]

STATE_ORDER = [
    "MS1",
    "MS2",
    "MS3",
    "MS4",
    "MS5",
    "MS6",
    "MS7",
    "MS8"
]


availability_rows = []

for i in ms3_idx:

    local_states = (
        state_array[
            nbr_idx[i]
        ]
    )

    row = {
        "cell":
            adata34.obs_names[i],

        "patient":
            patient_array[i]
    }

    for state in STATE_ORDER:

        row[
            f"neighbor_fraction_{state}"
        ] = float(
            np.mean(
                local_states
                ==
                state
            )
        )

    availability_rows.append(
        row
    )


availability_cell = pd.DataFrame(
    availability_rows
)

availability_cell.to_csv(
    RESULTS /
    "Step34C_MS3_LocalDestinationAvailability_CellLevel.csv",
    index=False
)


availability_patient = (
    availability_cell
    .groupby(
        "patient",
        as_index=False
    )
    .median(
        numeric_only=True
    )
)

availability_patient.to_csv(
    RESULTS /
    "Step34C_MS3_LocalDestinationAvailability_PatientLevel.csv",
    index=False
)


print(
    "\nMS3 local-neighborhood destination fractions:"
)

cols_show = [
    "patient",
    "neighbor_fraction_MS3",
    "neighbor_fraction_MS5",
    "neighbor_fraction_MS6"
]

print(
    availability_patient[
        cols_show
    ].to_string(
        index=False
    )
)

print()
print("=" * 90)
print("Cell 12 complete")
print("=" * 90)

In [ ]:
# Cell 13B — Recover FOXM1 native transition pilot without rerunning simulation

print("=" * 90)
print("Cell 13B — FOXM1 native transition probability recovery")
print("=" * 90)

import numpy as np
import pandas as pd
import scipy.sparse as sp

# ============================================================
# Required CellOracle outputs from Cell 13
# ============================================================

required_attrs = [
    "corrcoef",
    "embedding_knn",
]

for attr in required_attrs:
    assert hasattr(
        oracle34,
        attr
    ), f"Missing oracle34.{attr}"

HAS_RANDOM = hasattr(
    oracle34,
    "corrcoef_random"
)

print(
    "corrcoef:",
    type(oracle34.corrcoef),
    np.shape(oracle34.corrcoef)
)

print(
    "corrcoef_random:",
    (
        type(oracle34.corrcoef_random)
        if HAS_RANDOM
        else None
    ),
    (
        np.shape(oracle34.corrcoef_random)
        if HAS_RANDOM
        else None
    )
)

print(
    "embedding_knn:",
    type(oracle34.embedding_knn),
    oracle34.embedding_knn.shape
)

print(
    "embedding_knn nnz:",
    oracle34.embedding_knn.nnz
)

print(
    "randomized control available:",
    HAS_RANDOM
)

assert HAS_RANDOM


# ============================================================
# Frozen metadata
# ============================================================

state_array = (
    adata34.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
    .values
)

patient_array = (
    adata34.obs[
        "patient"
    ]
    .astype(str)
    .values
)

STATE_ORDER = [
    "MS1",
    "MS2",
    "MS3",
    "MS4",
    "MS5",
    "MS6",
    "MS7",
    "MS8",
]

ms3_idx = np.where(
    state_array == "MS3"
)[0]

assert len(ms3_idx) == 720


# ============================================================
# Helpers
# ============================================================

knn_csr = (
    oracle34.embedding_knn
    .tocsr()
)


def get_corr_values(
    corr,
    row_idx,
    col_idx
):
    """
    Extract correlation values for one cell and its
    actual frozen embedding neighbors.
    """

    if sp.issparse(corr):

        return np.asarray(
            corr.getrow(
                row_idx
            )[:, col_idx]
            .toarray()
        ).ravel()

    return np.asarray(
        corr[
            row_idx,
            col_idx
        ]
    ).ravel()


def stable_transition_weights(
    corr_values,
    sigma_corr=0.05
):
    """
    Mathematically equivalent to:
        exp(corr / sigma_corr) / sum(exp(...))
    but numerically stable.
    """

    logits = (
        np.asarray(
            corr_values,
            dtype=np.float64
        )
        /
        sigma_corr
    )

    logits -= np.max(logits)

    weights = np.exp(
        logits
    )

    total = weights.sum()

    assert (
        np.isfinite(total)
        and
        total > 0
    )

    return (
        weights
        /
        total
    )


# ============================================================
# MS3-only transition probabilities
# ============================================================

SIGMA_CORR = 0.05

rows = []

for i in ms3_idx:

    start = knn_csr.indptr[i]
    end = knn_csr.indptr[i + 1]

    neighbors = (
        knn_csr.indices[
            start:end
        ]
    )

    # safety: exclude self if present
    neighbors = neighbors[
        neighbors != i
    ]

    assert len(neighbors) > 0

    corr_real = get_corr_values(
        oracle34.corrcoef,
        i,
        neighbors
    )

    corr_rand = get_corr_values(
        oracle34.corrcoef_random,
        i,
        neighbors
    )

    w_real = stable_transition_weights(
        corr_real,
        SIGMA_CORR
    )

    w_rand = stable_transition_weights(
        corr_rand,
        SIGMA_CORR
    )

    # technical normalization check
    assert np.isclose(
        w_real.sum(),
        1.0,
        atol=1e-10
    )

    assert np.isclose(
        w_rand.sum(),
        1.0,
        atol=1e-10
    )

    nbr_states = (
        state_array[
            neighbors
        ]
    )

    row = {
        "cell":
            str(
                adata34.obs_names[i]
            ),

        "patient":
            patient_array[i],

        "source_state":
            "MS3",

        "n_neighbors":
            int(
                len(neighbors)
            ),
    }

    for state in STATE_ORDER:

        mask = (
            nbr_states == state
        )

        # local structural availability
        availability = float(
            np.mean(mask)
        )

        # CellOracle real transition mass
        real_mass = float(
            w_real[mask].sum()
        )

        # randomized residual negative-control mass
        random_mass = float(
            w_rand[mask].sum()
        )

        # enrichment relative to local neighborhood availability
        real_bias = (
            real_mass
            -
            availability
        )

        random_bias = (
            random_mass
            -
            availability
        )

        # perturbation-specific excess over randomized control
        excess_vs_random = (
            real_mass
            -
            random_mass
        )

        row[
            f"{state}_availability"
        ] = availability

        row[
            f"{state}_transition_mass"
        ] = real_mass

        row[
            f"{state}_random_mass"
        ] = random_mass

        row[
            f"{state}_transition_bias"
        ] = real_bias

        row[
            f"{state}_random_bias"
        ] = random_bias

        row[
            f"{state}_excess_vs_random"
        ] = excess_vs_random

    rows.append(row)


foxm1_transition_cell = (
    pd.DataFrame(rows)
)


# ============================================================
# Global technical QC
# ============================================================

real_mass_cols = [
    f"{state}_transition_mass"
    for state in STATE_ORDER
]

random_mass_cols = [
    f"{state}_random_mass"
    for state in STATE_ORDER
]

real_sums = (
    foxm1_transition_cell[
        real_mass_cols
    ]
    .sum(axis=1)
)

random_sums = (
    foxm1_transition_cell[
        random_mass_cols
    ]
    .sum(axis=1)
)

print()
print(
    "MS3 cells:",
    len(
        foxm1_transition_cell
    )
)

print(
    "Neighbor count min/median/max:",
    int(
        foxm1_transition_cell[
            "n_neighbors"
        ].min()
    ),
    float(
        foxm1_transition_cell[
            "n_neighbors"
        ].median()
    ),
    int(
        foxm1_transition_cell[
            "n_neighbors"
        ].max()
    ),
)

print(
    "Real state-mass sum min/median/max:",
    float(real_sums.min()),
    float(real_sums.median()),
    float(real_sums.max()),
)

print(
    "Random state-mass sum min/median/max:",
    float(random_sums.min()),
    float(random_sums.median()),
    float(random_sums.max()),
)

assert np.allclose(
    real_sums.values,
    1.0,
    atol=1e-8
)

assert np.allclose(
    random_sums.values,
    1.0,
    atol=1e-8
)


# ============================================================
# Save
# ============================================================

FOXM1_CELL_PATH = (
    RESULTS /
    "Step34C_FOXM1_MS3_NativeTransition_CellLevel.csv"
)

foxm1_transition_cell.to_csv(
    FOXM1_CELL_PATH,
    index=False
)

print()
print("Saved:")
print(FOXM1_CELL_PATH)

print()
print("=" * 90)
print("Cell 13B complete")
print("=" * 90)

In [ ]:
# Cell 14 — FOXM1 native transition pilot REPORT

print("=" * 90)
print("STEP34C — FOXM1 NATIVE TRANSITION PILOT REPORT")
print("=" * 90)

PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10",
]

FOCUS_STATES = [
    "MS3",
    "MS5",
    "MS6",
]


# ============================================================
# Patient-level medians
# ============================================================

metric_cols = []

for state in FOCUS_STATES:

    for metric in [
        "availability",
        "transition_mass",
        "transition_bias",
        "excess_vs_random",
    ]:

        metric_cols.append(
            f"{state}_{metric}"
        )


foxm1_patient = (
    foxm1_transition_cell
    .groupby(
        "patient",
        as_index=False
    )[
        metric_cols
    ]
    .median()
)

foxm1_patient[
    "patient_role"
] = np.where(
    foxm1_patient[
        "patient"
    ].isin(
        PRIMARY_PATIENTS
    ),
    "primary_replication",
    "sparse_sensitivity"
)

FOXM1_PATIENT_PATH = (
    RESULTS /
    "Step34C_FOXM1_MS3_NativeTransition_PatientLevel.csv"
)

foxm1_patient.to_csv(
    FOXM1_PATIENT_PATH,
    index=False
)


# ============================================================
# Print compact report
# ============================================================

show_cols = [
    "patient",
    "patient_role",

    "MS3_transition_mass",
    "MS3_transition_bias",
    "MS3_excess_vs_random",

    "MS5_transition_mass",
    "MS5_transition_bias",
    "MS5_excess_vs_random",

    "MS6_transition_mass",
    "MS6_transition_bias",
    "MS6_excess_vs_random",
]

print()
print(
    foxm1_patient[
        show_cols
    ].to_string(
        index=False
    )
)


# ============================================================
# Primary-patient direction summary
# ============================================================

primary = foxm1_patient.loc[
    foxm1_patient[
        "patient"
    ].isin(
        PRIMARY_PATIENTS
    )
].copy()

direction_summary = {
    "MS3_excess_negative_n":
        int(
            (
                primary[
                    "MS3_excess_vs_random"
                ]
                < 0
            ).sum()
        ),

    "MS5_excess_positive_n":
        int(
            (
                primary[
                    "MS5_excess_vs_random"
                ]
                > 0
            ).sum()
        ),

    "MS6_excess_positive_n":
        int(
            (
                primary[
                    "MS6_excess_vs_random"
                ]
                > 0
            ).sum()
        ),
}

print()
print("Primary-patient directional summary:")
print(
    json.dumps(
        direction_summary,
        indent=2
    )
)


# ============================================================
# Save pilot lock
# ============================================================

pilot_report = {
    "status":
        "FOXM1_NATIVE_TRANSITION_PILOT_COMPLETE",

    "source_state":
        "MS3",

    "formal_TF":
        "FOXM1",

    "sigma_corr":
        SIGMA_CORR,

    "transition_method":
        (
            "CellOracle native correlation-derived "
            "transition probability aggregated over "
            "frozen MS1-MS8 destination labels"
        ),

    "full_delta_embedding_calculated":
        False,

    "reason_no_full_delta_embedding":
        (
            "Not required for state-level transition analysis "
            "and avoided because the frozen embedding has "
            "30 dimensions, making the full pairwise "
            "unitary-vector tensor memory-intensive."
        ),

    "primary_patients":
        PRIMARY_PATIENTS,

    "direction_summary":
        direction_summary,

    "claim_boundary":
        (
            "Predicted transition propensity within the frozen "
            "CellOracle neighborhood graph; not lineage conversion."
        )
}

FOXM1_REPORT_PATH = (
    LOGS /
    "Step34C_FOXM1_NativeTransition_Pilot.json"
)

with open(
    FOXM1_REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pilot_report,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("Saved:")
print(FOXM1_PATIENT_PATH)
print(FOXM1_REPORT_PATH)

print()
print("=" * 90)
print("FOXM1 NATIVE TRANSITION PILOT: COMPLETE")
print("=" * 90)

In [ ]:
# Cell 15 — FOXM1 native MS3-only embedding displacement

print("=" * 90)
print("Cell 15 — FOXM1 native MS3-only embedding displacement")
print("=" * 90)

import numpy as np
import pandas as pd
import scipy.sparse as sp

H = np.asarray(
    adata34.obsm[
        "X_pca_harmony_lineage_clean"
    ],
    dtype=np.float64
)

assert H.shape == (12057, 30)

STATE_ORDER = [
    "MS1", "MS2", "MS3", "MS4",
    "MS5", "MS6", "MS7", "MS8"
]

state_array = (
    adata34.obs[
        "final_malignant_state_id"
    ]
    .astype(str)
    .values
)

patient_array = (
    adata34.obs["patient"]
    .astype(str)
    .values
)

ms3_idx = np.where(
    state_array == "MS3"
)[0]

assert len(ms3_idx) == 720


# ============================================================
# Frozen state centroids
# ============================================================

state_centroids = {}

for state in STATE_ORDER:

    mask = (
        state_array == state
    )

    state_centroids[state] = (
        H[mask].mean(axis=0)
    )

print("Frozen state centroids built.")


# ============================================================
# CellOracle ingredients
# ============================================================

assert hasattr(
    oracle34,
    "corrcoef"
)

assert hasattr(
    oracle34,
    "corrcoef_random"
)

assert hasattr(
    oracle34,
    "embedding_knn"
)

knn_csr = (
    oracle34.embedding_knn
    .tocsr()
)

SIGMA_CORR = 0.05


def get_corr_values(
    corr,
    row_idx,
    col_idx
):

    if sp.issparse(corr):

        return np.asarray(
            corr.getrow(
                row_idx
            )[:, col_idx]
            .toarray()
        ).ravel()

    return np.asarray(
        corr[
            row_idx,
            col_idx
        ]
    ).ravel()


def stable_transition_weights(
    corr_values,
    sigma_corr=0.05
):

    logits = (
        np.asarray(
            corr_values,
            dtype=np.float64
        )
        /
        sigma_corr
    )

    logits -= np.max(logits)

    w = np.exp(logits)

    return w / w.sum()


# ============================================================
# Exact row-wise CellOracle embedding-shift formula
#
# delta =
# sum_j P_ij * (Y_j - Y_i)
# -
# mean_neighbor_j (Y_j - Y_i)
# ============================================================

rows = []

for i in ms3_idx:

    start = knn_csr.indptr[i]
    end = knn_csr.indptr[i + 1]

    neighbors = (
        knn_csr.indices[
            start:end
        ]
    )

    neighbors = neighbors[
        neighbors != i
    ]

    assert len(neighbors) > 0

    vectors = (
        H[neighbors]
        -
        H[i]
    )

    corr_real = get_corr_values(
        oracle34.corrcoef,
        i,
        neighbors
    )

    corr_rand = get_corr_values(
        oracle34.corrcoef_random,
        i,
        neighbors
    )

    w_real = stable_transition_weights(
        corr_real,
        SIGMA_CORR
    )

    w_rand = stable_transition_weights(
        corr_rand,
        SIGMA_CORR
    )

    local_baseline = (
        vectors.mean(axis=0)
    )

    delta_real = (
        (
            w_real[:, None]
            *
            vectors
        ).sum(axis=0)
        -
        local_baseline
    )

    delta_rand = (
        (
            w_rand[:, None]
            *
            vectors
        ).sum(axis=0)
        -
        local_baseline
    )

    y0 = H[i]

    y_real = (
        y0 + delta_real
    )

    y_rand = (
        y0 + delta_rand
    )

    row = {
        "cell":
            str(
                adata34.obs_names[i]
            ),

        "patient":
            patient_array[i],

        "delta_norm_real":
            float(
                np.linalg.norm(
                    delta_real
                )
            ),

        "delta_norm_random":
            float(
                np.linalg.norm(
                    delta_rand
                )
            ),
    }


    # ========================================================
    # Frozen-state geometric readouts
    # ========================================================

    for target in [
        "MS3",
        "MS5",
        "MS6"
    ]:

        centroid = (
            state_centroids[target]
        )

        target_vec = (
            centroid - y0
        )

        target_dist = (
            np.linalg.norm(
                target_vec
            )
        )

        if target_dist > 1e-12:

            target_unit = (
                target_vec
                /
                target_dist
            )

            projection_real = float(
                np.dot(
                    delta_real,
                    target_unit
                )
            )

            projection_random = float(
                np.dot(
                    delta_rand,
                    target_unit
                )
            )

        else:

            projection_real = np.nan
            projection_random = np.nan


        dist_real = float(
            np.linalg.norm(
                centroid
                -
                y_real
            )
        )

        dist_random = float(
            np.linalg.norm(
                centroid
                -
                y_rand
            )
        )


        row[
            f"{target}_baseline_distance"
        ] = float(
            target_dist
        )

        row[
            f"{target}_projection_real"
        ] = projection_real

        row[
            f"{target}_projection_random"
        ] = projection_random

        row[
            f"{target}_projection_excess"
        ] = (
            projection_real
            -
            projection_random
        )

        row[
            f"{target}_distance_change_real"
        ] = (
            dist_real
            -
            target_dist
        )

        row[
            f"{target}_distance_change_random"
        ] = (
            dist_random
            -
            target_dist
        )

        row[
            f"{target}_distance_change_excess"
        ] = (
            (
                dist_real
                -
                target_dist
            )
            -
            (
                dist_random
                -
                target_dist
            )
        )


    rows.append(row)


foxm1_displacement = pd.DataFrame(
    rows
)

assert len(
    foxm1_displacement
) == 720


# ============================================================
# Save
# ============================================================

FOX_D_CELL = (
    RESULTS /
    "Step34D_FOXM1_MS3_NativeEmbeddingDisplacement_CellLevel.csv"
)

foxm1_displacement.to_csv(
    FOX_D_CELL,
    index=False
)

print(
    "MS3 displacement rows:",
    len(
        foxm1_displacement
    )
)

print()
print("Saved:")
print(FOX_D_CELL)

print()
print("=" * 90)
print("Cell 15 complete")
print("=" * 90)

In [ ]:
# Cell 16 — FOXM1 native state-space displacement REPORT

print("=" * 90)
print("STEP34D — FOXM1 STATE-SPACE DISPLACEMENT REPORT")
print("=" * 90)

PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

summary_cols = [
    "MS3_distance_change_excess",

    "MS5_projection_excess",
    "MS5_distance_change_excess",

    "MS6_projection_excess",
    "MS6_distance_change_excess",

    "delta_norm_real",
    "delta_norm_random",
]


foxm1_disp_patient = (
    foxm1_displacement
    .groupby(
        "patient",
        as_index=False
    )[
        summary_cols
    ]
    .median()
)


foxm1_disp_patient[
    "patient_role"
] = np.where(
    foxm1_disp_patient[
        "patient"
    ].isin(
        PRIMARY_PATIENTS
    ),
    "primary_replication",
    "sparse_sensitivity"
)


FOX_D_PATIENT = (
    RESULTS /
    "Step34D_FOXM1_MS3_NativeEmbeddingDisplacement_PatientLevel.csv"
)

foxm1_disp_patient.to_csv(
    FOX_D_PATIENT,
    index=False
)


print()
print(
    foxm1_disp_patient[
        [
            "patient",
            "patient_role",

            "MS3_distance_change_excess",

            "MS5_projection_excess",
            "MS5_distance_change_excess",

            "MS6_projection_excess",
            "MS6_distance_change_excess",
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# Primary-patient directional consistency
# ============================================================

primary = foxm1_disp_patient.loc[
    foxm1_disp_patient[
        "patient"
    ].isin(
        PRIMARY_PATIENTS
    )
].copy()


direction = {
    "away_from_MS3_n":
        int(
            (
                primary[
                    "MS3_distance_change_excess"
                ]
                > 0
            ).sum()
        ),

    "toward_MS5_projection_n":
        int(
            (
                primary[
                    "MS5_projection_excess"
                ]
                > 0
            ).sum()
        ),

    "closer_to_MS5_n":
        int(
            (
                primary[
                    "MS5_distance_change_excess"
                ]
                < 0
            ).sum()
        ),

    "toward_MS6_projection_n":
        int(
            (
                primary[
                    "MS6_projection_excess"
                ]
                > 0
            ).sum()
        ),

    "closer_to_MS6_n":
        int(
            (
                primary[
                    "MS6_distance_change_excess"
                ]
                < 0
            ).sum()
        ),
}


print()
print(
    "Primary-patient directional consistency:"
)

print(
    json.dumps(
        direction,
        indent=2
    )
)


# ============================================================
# Combine with first-hop pilot interpretation
# ============================================================

formal_pilot = {
    "status":
        "FOXM1_STATE_DISPLACEMENT_PILOT_COMPLETE",

    "formal_method":
        (
            "MS3-only row-wise implementation of the "
            "native CellOracle embedding-shift formula"
        ),

    "frozen_embedding":
        "X_pca_harmony_lineage_clean",

    "embedding_dimensions":
        30,

    "sigma_corr":
        0.05,

    "state_centroids_redefined":
        False,

    "states_reclustered":
        False,

    "directional_consistency":
        direction,

    "first_hop_transition_interpretation": {
        "MS3_retention":
            "3/3 primary patients showed negative excess versus randomized control",

        "MS5":
            (
                "Positive in HCC09 and HCC10; "
                "HCC08 structurally non-evaluable because "
                "local MS5 availability was zero"
            ),

        "MS6":
            (
                "Direct first-hop MS3-to-MS6 transition "
                "was structurally non-evaluable in all "
                "three primary patients at k=200"
            ),
    },

    "claim_boundary":
        (
            "Counterfactual local state-space displacement "
            "within the frozen CellOracle/Harmony geometry; "
            "not lineage conversion."
        ),
}


FOX_D_REPORT = (
    LOGS /
    "Step34D_FOXM1_StateDisplacement_Pilot.json"
)

with open(
    FOX_D_REPORT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        formal_pilot,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("Saved:")
print(FOX_D_PATIENT)
print(FOX_D_REPORT)

print()
print("=" * 90)
print("FOXM1 STATE-SPACE DISPLACEMENT PILOT: COMPLETE")
print("=" * 90)

## Step34E. Frozen MS3–MS5–MS6 Axis Audit and Five-Regulator Integration

FOXM1 showed reproducible counterfactual displacement away from MS3 and toward
the frozen MS5/MS6 region across HCC08, HCC09, and HCC10.

Before extending the analysis to the remaining regulators, the geometry of the frozen
MS3–MS5–MS6 axis is audited to determine whether MS5- and MS6-directed displacement
represent distinct directions or a shared differentiation-associated axis.

The remaining four frozen regulators are then evaluated using the identical native
CellOracle transition and embedding-shift framework.

In [ ]:
# Cell 17 — Frozen MS3-MS5-MS6 axis geometry audit

print("=" * 90)
print("Cell 17 — Frozen MS3-MS5-MS6 axis geometry audit")
print("=" * 90)

import numpy as np
import pandas as pd
import json

C3 = state_centroids["MS3"]
C5 = state_centroids["MS5"]
C6 = state_centroids["MS6"]

v35 = C5 - C3
v36 = C6 - C3
v56 = C6 - C5

u35 = v35 / np.linalg.norm(v35)
u36 = v36 / np.linalg.norm(v36)
u56 = v56 / np.linalg.norm(v56)

cos_35_36 = float(
    np.dot(
        u35,
        u36
    )
)

cos_35_56 = float(
    np.dot(
        u35,
        u56
    )
)

cos_36_56 = float(
    np.dot(
        u36,
        u56
    )
)

dist_35 = float(
    np.linalg.norm(v35)
)

dist_36 = float(
    np.linalg.norm(v36)
)

dist_56 = float(
    np.linalg.norm(v56)
)


axis_geometry = pd.DataFrame([
    {
        "metric": "distance_MS3_MS5",
        "value": dist_35
    },
    {
        "metric": "distance_MS3_MS6",
        "value": dist_36
    },
    {
        "metric": "distance_MS5_MS6",
        "value": dist_56
    },
    {
        "metric": "cosine_MS3toMS5_vs_MS3toMS6",
        "value": cos_35_36
    },
    {
        "metric": "cosine_MS3toMS5_vs_MS5toMS6",
        "value": cos_35_56
    },
    {
        "metric": "cosine_MS3toMS6_vs_MS5toMS6",
        "value": cos_36_56
    },
])

axis_geometry.to_csv(
    RESULTS /
    "Step34E_Frozen_MS3_MS5_MS6_AxisGeometry.csv",
    index=False
)

print(axis_geometry.to_string(index=False))

print()
print(
    "MS3→MS5 vs MS3→MS6 cosine:",
    round(cos_35_36, 4)
)

if cos_35_36 >= 0.8:
    interpretation = (
        "MS5- and MS6-directed displacement are strongly aligned "
        "and should primarily be interpreted as movement along a "
        "shared MS3-to-differentiated-state axis."
    )
elif cos_35_36 >= 0.5:
    interpretation = (
        "MS5- and MS6-directed displacement are moderately aligned "
        "but retain partially distinct geometry."
    )
else:
    interpretation = (
        "MS5- and MS6-directed displacement represent substantially "
        "different geometric directions."
    )

print()
print("Interpretation:")
print(interpretation)

axis_report = {
    "status":
        "FROZEN_AXIS_GEOMETRY_AUDITED",

    "distance_MS3_MS5":
        dist_35,

    "distance_MS3_MS6":
        dist_36,

    "distance_MS5_MS6":
        dist_56,

    "cosine_MS3toMS5_vs_MS3toMS6":
        cos_35_36,

    "interpretation":
        interpretation,

    "claim_boundary":
        (
            "Geometric alignment within the frozen Harmony space; "
            "not lineage topology."
        )
}

with open(
    LOGS /
    "Step34E_FrozenAxisGeometry.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        axis_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("=" * 90)
print("Cell 17 complete")
print("=" * 90)

In [ ]:
# Cell 18 — Unified native CellOracle state-transition function

print("=" * 90)
print("Cell 18 — Build unified Step34 transition function")
print("=" * 90)

import scipy.sparse as sp


PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10"
]

FORMAL_TFS = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]

STATE_ORDER = [
    "MS1",
    "MS2",
    "MS3",
    "MS4",
    "MS5",
    "MS6",
    "MS7",
    "MS8"
]

SIGMA_CORR = 0.05
N_TRANSITION_NEIGHBORS = 200


def _corr_values(
    corr,
    row_idx,
    col_idx
):

    if sp.issparse(corr):

        return np.asarray(
            corr.getrow(
                row_idx
            )[:, col_idx]
            .toarray()
        ).ravel()

    return np.asarray(
        corr[
            row_idx,
            col_idx
        ]
    ).ravel()


def _stable_weights(
    corr_values,
    sigma_corr=0.05
):

    logits = (
        np.asarray(
            corr_values,
            dtype=np.float64
        )
        /
        sigma_corr
    )

    logits -= np.max(logits)

    w = np.exp(logits)

    return w / w.sum()


def score_current_native_transition(tf):

    print()
    print("=" * 90)
    print(f"Native transition scoring — {tf}")
    print("=" * 90)

    assert hasattr(
        oracle34,
        "corrcoef"
    )

    assert hasattr(
        oracle34,
        "corrcoef_random"
    )

    assert hasattr(
        oracle34,
        "embedding_knn"
    )

    knn = (
        oracle34.embedding_knn
        .tocsr()
    )

    cell_rows = []

    for i in ms3_idx:

        start = knn.indptr[i]
        end = knn.indptr[i + 1]

        neighbors = (
            knn.indices[
                start:end
            ]
        )

        neighbors = neighbors[
            neighbors != i
        ]

        assert len(neighbors) > 0

        corr_real = _corr_values(
            oracle34.corrcoef,
            i,
            neighbors
        )

        corr_rand = _corr_values(
            oracle34.corrcoef_random,
            i,
            neighbors
        )

        w_real = _stable_weights(
            corr_real,
            SIGMA_CORR
        )

        w_rand = _stable_weights(
            corr_rand,
            SIGMA_CORR
        )

        vectors = (
            H[neighbors]
            -
            H[i]
        )

        local_baseline = (
            vectors.mean(axis=0)
        )

        delta_real = (
            (
                w_real[:, None]
                *
                vectors
            ).sum(axis=0)
            -
            local_baseline
        )

        delta_rand = (
            (
                w_rand[:, None]
                *
                vectors
            ).sum(axis=0)
            -
            local_baseline
        )

        y0 = H[i]

        row = {
            "TF":
                tf,

            "cell":
                str(
                    adata34.obs_names[i]
                ),

            "patient":
                patient_array[i],

            "n_neighbors":
                int(
                    len(neighbors)
                ),

            "delta_norm_real":
                float(
                    np.linalg.norm(
                        delta_real
                    )
                ),

            "delta_norm_random":
                float(
                    np.linalg.norm(
                        delta_rand
                    )
                ),
        }

        nbr_states = (
            state_array[
                neighbors
            ]
        )

        # --------------------------------------------------
        # First-hop transition mass
        # --------------------------------------------------

        for state in STATE_ORDER:

            state_mask = (
                nbr_states
                ==
                state
            )

            availability = float(
                np.mean(
                    state_mask
                )
            )

            real_mass = float(
                w_real[
                    state_mask
                ].sum()
            )

            random_mass = float(
                w_rand[
                    state_mask
                ].sum()
            )

            row[
                f"{state}_availability"
            ] = availability

            row[
                f"{state}_transition_mass"
            ] = real_mass

            row[
                f"{state}_random_mass"
            ] = random_mass

            row[
                f"{state}_transition_bias"
            ] = (
                real_mass
                -
                availability
            )

            row[
                f"{state}_excess_vs_random"
            ] = (
                real_mass
                -
                random_mass
            )

        # --------------------------------------------------
        # Native embedding displacement
        # --------------------------------------------------

        for target in [
            "MS3",
            "MS5",
            "MS6"
        ]:

            centroid = (
                state_centroids[
                    target
                ]
            )

            target_vec = (
                centroid
                -
                y0
            )

            target_dist = float(
                np.linalg.norm(
                    target_vec
                )
            )

            if target_dist > 1e-12:

                target_unit = (
                    target_vec
                    /
                    target_dist
                )

                projection_real = float(
                    np.dot(
                        delta_real,
                        target_unit
                    )
                )

                projection_random = float(
                    np.dot(
                        delta_rand,
                        target_unit
                    )
                )

            else:

                projection_real = np.nan
                projection_random = np.nan

            y_real = (
                y0
                +
                delta_real
            )

            y_rand = (
                y0
                +
                delta_rand
            )

            dist_real = float(
                np.linalg.norm(
                    centroid
                    -
                    y_real
                )
            )

            dist_rand = float(
                np.linalg.norm(
                    centroid
                    -
                    y_rand
                )
            )

            row[
                f"{target}_projection_real"
            ] = projection_real

            row[
                f"{target}_projection_random"
            ] = projection_random

            row[
                f"{target}_projection_excess"
            ] = (
                projection_real
                -
                projection_random
            )

            row[
                f"{target}_distance_change_real"
            ] = (
                dist_real
                -
                target_dist
            )

            row[
                f"{target}_distance_change_random"
            ] = (
                dist_rand
                -
                target_dist
            )

            row[
                f"{target}_distance_change_excess"
            ] = (
                (
                    dist_real
                    -
                    target_dist
                )
                -
                (
                    dist_rand
                    -
                    target_dist
                )
            )

        # --------------------------------------------------
        # Explicit shared-axis projections
        # --------------------------------------------------

        row[
            "MS3_to_MS5_axis_projection_excess"
        ] = float(
            np.dot(
                delta_real
                -
                delta_rand,
                u35
            )
        )

        row[
            "MS3_to_MS6_axis_projection_excess"
        ] = float(
            np.dot(
                delta_real
                -
                delta_rand,
                u36
            )
        )

        cell_rows.append(
            row
        )

    cell_df = pd.DataFrame(
        cell_rows
    )

    assert len(cell_df) == 720

    # --------------------------------------------------
    # Patient-level medians
    # --------------------------------------------------

    patient_cols = [
        "MS3_excess_vs_random",

        "MS5_availability",
        "MS5_excess_vs_random",

        "MS6_availability",
        "MS6_excess_vs_random",

        "MS3_distance_change_excess",

        "MS5_projection_excess",
        "MS5_distance_change_excess",

        "MS6_projection_excess",
        "MS6_distance_change_excess",

        "MS3_to_MS5_axis_projection_excess",
        "MS3_to_MS6_axis_projection_excess",
    ]

    patient_df = (
        cell_df
        .groupby(
            "patient",
            as_index=False
        )[
            patient_cols
        ]
        .median()
    )

    patient_df[
        "patient_role"
    ] = np.where(
        patient_df[
            "patient"
        ].isin(
            PRIMARY_PATIENTS
        ),
        "primary_replication",
        "sparse_sensitivity"
    )

    # --------------------------------------------------
    # Save immediately before next TF overwrites simulation
    # --------------------------------------------------

    cell_path = (
        RESULTS /
        f"Step34E_{tf}_MS3_NativeStateTransition_CellLevel.csv"
    )

    patient_path = (
        RESULTS /
        f"Step34E_{tf}_MS3_NativeStateTransition_PatientLevel.csv"
    )

    cell_df.to_csv(
        cell_path,
        index=False
    )

    patient_df.to_csv(
        patient_path,
        index=False
    )

    # --------------------------------------------------
    # Primary directions
    # --------------------------------------------------

    prim = patient_df.loc[
        patient_df[
            "patient"
        ].isin(
            PRIMARY_PATIENTS
        )
    ].copy()

    direction = {
        "away_from_MS3_n":
            int(
                (
                    prim[
                        "MS3_distance_change_excess"
                    ]
                    > 0
                ).sum()
            ),

        "toward_MS5_n":
            int(
                (
                    prim[
                        "MS5_projection_excess"
                    ]
                    > 0
                ).sum()
            ),

        "closer_to_MS5_n":
            int(
                (
                    prim[
                        "MS5_distance_change_excess"
                    ]
                    < 0
                ).sum()
            ),

        "toward_MS6_n":
            int(
                (
                    prim[
                        "MS6_projection_excess"
                    ]
                    > 0
                ).sum()
            ),

        "closer_to_MS6_n":
            int(
                (
                    prim[
                        "MS6_distance_change_excess"
                    ]
                    < 0
                ).sum()
            ),

        "positive_MS3_to_MS5_axis_n":
            int(
                (
                    prim[
                        "MS3_to_MS5_axis_projection_excess"
                    ]
                    > 0
                ).sum()
            ),

        "positive_MS3_to_MS6_axis_n":
            int(
                (
                    prim[
                        "MS3_to_MS6_axis_projection_excess"
                    ]
                    > 0
                ).sum()
            ),
    }

    return (
        cell_df,
        patient_df,
        direction
    )


print("Unified Step34 transition function ready.")

print("=" * 90)

In [ ]:
# Cell 19 — Run remaining four formal regulators

print("=" * 90)
print("Cell 19 — Remaining four formal regulator perturbations")
print("=" * 90)

REMAINING_TFS = [
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]

batch_direction = {}

for tf in REMAINING_TFS:

    print()
    print("#" * 90)
    print(f"FORMAL STEP34 — {tf}")
    print("#" * 90)

    assert (
        tf
        in
        oracle34.active_regulatory_genes
    )

    # ========================================================
    # Same frozen Step33 perturbation definition
    # ========================================================

    oracle34.simulate_shift(
        perturb_condition={
            tf: 0.0
        },
        n_propagation=3,
        use_randomized_GRN=False,
        clip_delta_X=False
    )

    print(
        f"{tf}: simulation complete"
    )

    # ========================================================
    # Native CellOracle transition correlation
    # ========================================================

    oracle34.estimate_transition_prob(
        n_neighbors=200,
        knn_random=True,
        sampled_fraction=1,
        n_jobs=4,
        calculate_randomized=True,
        random_seed=15071990
    )

    print(
        f"{tf}: native transition correlation complete"
    )

    # ========================================================
    # Score and SAVE before next TF
    # ========================================================

    cell_df, patient_df, direction = (
        score_current_native_transition(
            tf
        )
    )

    batch_direction[tf] = (
        direction
    )

    print()
    print(
        patient_df[
            [
                "patient",
                "patient_role",
                "MS3_distance_change_excess",
                "MS5_projection_excess",
                "MS5_distance_change_excess",
                "MS6_projection_excess",
                "MS6_distance_change_excess",
            ]
        ].to_string(
            index=False
        )
    )

    print()
    print(
        "Direction:",
        direction
    )


print()
print("=" * 90)
print("Remaining four TFs complete")
print("=" * 90)

In [ ]:
# Cell 20 — Five-TF Step34 integrated report

print("=" * 90)
print("STEP34E — FIVE-TF STATE-TRANSITION INTEGRATED REPORT")
print("=" * 90)

# ============================================================
# Load FOXM1 formal pilot
# ============================================================

fox_path = (
    RESULTS /
    "Step34D_FOXM1_MS3_NativeEmbeddingDisplacement_PatientLevel.csv"
)

fox = pd.read_csv(
    fox_path
)

# Add axis metrics from FOXM1 cell-level file
fox_cell = pd.read_csv(
    RESULTS /
    "Step34D_FOXM1_MS3_NativeEmbeddingDisplacement_CellLevel.csv"
)

# Reconstruct shared-axis metrics from frozen delta outputs
# using projection_excess values already equivalent to the
# cell-specific centroid direction endpoints.
# Formal cross-TF axis-specific metrics are available for
# remaining TFs; FOXM1 is retained using its existing geometric endpoints.

frames = []

fox["TF"] = "FOXM1"

frames.append(
    fox
)


# ============================================================
# Remaining TFs
# ============================================================

for tf in [
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1"
]:

    df = pd.read_csv(
        RESULTS /
        f"Step34E_{tf}_MS3_NativeStateTransition_PatientLevel.csv"
    )

    df["TF"] = tf

    frames.append(
        df
    )


integrated_patient = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)

integrated_patient.to_csv(
    RESULTS /
    "Step34E_5TF_StateTransition_PatientLevel.csv",
    index=False
)


# ============================================================
# Primary patients only
# ============================================================

primary5 = integrated_patient.loc[
    integrated_patient[
        "patient"
    ].isin(
        PRIMARY_PATIENTS
    )
].copy()


direction_rows = []

for tf in FORMAL_TFS:

    sub = primary5.loc[
        primary5[
            "TF"
        ] == tf
    ]

    assert len(sub) == 3

    direction_rows.append({
        "TF":
            tf,

        "away_from_MS3":
            f"{int((sub['MS3_distance_change_excess'] > 0).sum())}/3",

        "toward_MS5":
            f"{int((sub['MS5_projection_excess'] > 0).sum())}/3",

        "closer_to_MS5":
            f"{int((sub['MS5_distance_change_excess'] < 0).sum())}/3",

        "toward_MS6":
            f"{int((sub['MS6_projection_excess'] > 0).sum())}/3",

        "closer_to_MS6":
            f"{int((sub['MS6_distance_change_excess'] < 0).sum())}/3",
    })


direction_df = pd.DataFrame(
    direction_rows
)

direction_df.to_csv(
    RESULTS /
    "Step34E_5TF_PrimaryPatient_DirectionConsistency.csv",
    index=False
)


print("\nPrimary-patient direction consistency:\n")

print(
    direction_df.to_string(
        index=False
    )
)


# ============================================================
# Cross-TF convergence
# ============================================================

convergence = {
    "TFs_away_from_MS3_3of3":
        int(
            (
                direction_df[
                    "away_from_MS3"
                ]
                ==
                "3/3"
            ).sum()
        ),

    "TFs_toward_MS5_3of3":
        int(
            (
                direction_df[
                    "toward_MS5"
                ]
                ==
                "3/3"
            ).sum()
        ),

    "TFs_closer_to_MS5_3of3":
        int(
            (
                direction_df[
                    "closer_to_MS5"
                ]
                ==
                "3/3"
            ).sum()
        ),

    "TFs_toward_MS6_3of3":
        int(
            (
                direction_df[
                    "toward_MS6"
                ]
                ==
                "3/3"
            ).sum()
        ),

    "TFs_closer_to_MS6_3of3":
        int(
            (
                direction_df[
                    "closer_to_MS6"
                ]
                ==
                "3/3"
            ).sum()
        ),
}


print()
print("Cross-TF convergence:")
print(
    json.dumps(
        convergence,
        indent=2
    )
)


report = {
    "status":
        "STEP34E_FIVE_TF_TRANSITION_COMPLETE",

    "formal_TFs":
        FORMAL_TFS,

    "primary_patients":
        PRIMARY_PATIENTS,

    "primary_state":
        "MS3",

    "frozen_target_states":
        [
            "MS5",
            "MS6"
        ],

    "cross_TF_convergence":
        convergence,

    "claim_boundary":
        (
            "Counterfactual state-space displacement "
            "within frozen CellOracle/Harmony geometry; "
            "not lineage conversion."
        )
}

with open(
    LOGS /
    "Step34E_5TF_StateTransition_Report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 90)
print("STEP34E STATUS: COMPLETE")
print("=" * 90)

## Step34F. Final Integration and Lock

Primary conclusion:

Virtual suppression of all five frozen MS3-associated regulators produced reproducible
counterfactual displacement away from the frozen MS3 state and toward the frozen MS5
intermediate-state region across HCC08, HCC09, and HCC10.

MS6-directed displacement is retained as secondary geometric context because MS3→MS5
and MS3→MS6 directions are strongly aligned in the frozen Harmony space, while MS6
proximity responses are less uniformly reproduced across regulators.

These results describe counterfactual state-space displacement, not lineage conversion.

In [ ]:
# Cell 21 — Formal Step34 patient-level integration

print("=" * 90)
print("Cell 21 — Formal Step34 patient-level integration")
print("=" * 90)

import pandas as pd
import numpy as np
import json

TF_ORDER = [
    "FOXM1",
    "MYBL2",
    "E2F8",
    "TFDP1",
    "E2F1",
]

PRIMARY_PATIENTS = [
    "HCC08",
    "HCC09",
    "HCC10",
]

frames = []

# ------------------------------------------------------------
# FOXM1
# ------------------------------------------------------------

fox = pd.read_csv(
    RESULTS /
    "Step34D_FOXM1_MS3_NativeEmbeddingDisplacement_PatientLevel.csv"
)

fox["TF"] = "FOXM1"

frames.append(fox)


# ------------------------------------------------------------
# Remaining four TFs
# ------------------------------------------------------------

for tf in TF_ORDER[1:]:

    df = pd.read_csv(
        RESULTS /
        f"Step34E_{tf}_MS3_NativeStateTransition_PatientLevel.csv"
    )

    df["TF"] = tf

    frames.append(df)


patient_all = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)

patient_primary = patient_all.loc[
    patient_all["patient"].isin(
        PRIMARY_PATIENTS
    )
].copy()

assert len(patient_primary) == 15


# ------------------------------------------------------------
# Formal primary endpoints
# ------------------------------------------------------------

patient_primary[
    "away_from_MS3"
] = (
    patient_primary[
        "MS3_distance_change_excess"
    ] > 0
)

patient_primary[
    "toward_MS5"
] = (
    patient_primary[
        "MS5_projection_excess"
    ] > 0
)

patient_primary[
    "closer_to_MS5"
] = (
    patient_primary[
        "MS5_distance_change_excess"
    ] < 0
)

# Secondary context only
patient_primary[
    "toward_MS6_secondary"
] = (
    patient_primary[
        "MS6_projection_excess"
    ] > 0
)

patient_primary[
    "closer_to_MS6_secondary"
] = (
    patient_primary[
        "MS6_distance_change_excess"
    ] < 0
)


formal_cols = [
    "TF",
    "patient",

    "MS3_distance_change_excess",

    "MS5_projection_excess",
    "MS5_distance_change_excess",

    "MS6_projection_excess",
    "MS6_distance_change_excess",

    "away_from_MS3",
    "toward_MS5",
    "closer_to_MS5",

    "toward_MS6_secondary",
    "closer_to_MS6_secondary",
]

formal_patient_table = (
    patient_primary[
        formal_cols
    ]
    .copy()
)

formal_patient_table.to_csv(
    RESULTS /
    "Step34_FINAL_5TF_PrimaryPatient_StateDisplacement.csv",
    index=False
)


print(
    formal_patient_table.to_string(
        index=False
    )
)

print()
print("=" * 90)
print("Cell 21 complete")
print("=" * 90)

In [ ]:
# Cell 22 — Cross-regulator convergence

print("=" * 90)
print("Cell 22 — Cross-regulator convergence")
print("=" * 90)

rows = []

for tf in TF_ORDER:

    sub = patient_primary.loc[
        patient_primary["TF"] == tf
    ]

    assert len(sub) == 3

    rows.append({
        "TF": tf,

        "away_from_MS3":
            f"{int(sub['away_from_MS3'].sum())}/3",

        "toward_MS5":
            f"{int(sub['toward_MS5'].sum())}/3",

        "closer_to_MS5":
            f"{int(sub['closer_to_MS5'].sum())}/3",

        "toward_MS6_secondary":
            f"{int(sub['toward_MS6_secondary'].sum())}/3",

        "closer_to_MS6_secondary":
            f"{int(sub['closer_to_MS6_secondary'].sum())}/3",
    })

direction_final = pd.DataFrame(rows)

direction_final.to_csv(
    RESULTS /
    "Step34_FINAL_5TF_DirectionConsistency.csv",
    index=False
)

print(
    direction_final.to_string(
        index=False
    )
)


cross_tf = pd.DataFrame([
    {
        "endpoint":
            "Away from MS3",

        "TFs_with_3of3_consistency":
            int(
                (
                    direction_final[
                        "away_from_MS3"
                    ] == "3/3"
                ).sum()
            ),

        "formal_TFs_n": 5,

        "role": "Primary"
    },

    {
        "endpoint":
            "Toward MS5",

        "TFs_with_3of3_consistency":
            int(
                (
                    direction_final[
                        "toward_MS5"
                    ] == "3/3"
                ).sum()
            ),

        "formal_TFs_n": 5,

        "role": "Primary"
    },

    {
        "endpoint":
            "Closer to MS5",

        "TFs_with_3of3_consistency":
            int(
                (
                    direction_final[
                        "closer_to_MS5"
                    ] == "3/3"
                ).sum()
            ),

        "formal_TFs_n": 5,

        "role": "Primary"
    },

    {
        "endpoint":
            "Toward MS6",

        "TFs_with_3of3_consistency":
            int(
                (
                    direction_final[
                        "toward_MS6_secondary"
                    ] == "3/3"
                ).sum()
            ),

        "formal_TFs_n": 5,

        "role": "Secondary geometric context"
    },

    {
        "endpoint":
            "Closer to MS6",

        "TFs_with_3of3_consistency":
            int(
                (
                    direction_final[
                        "closer_to_MS6_secondary"
                    ] == "3/3"
                ).sum()
            ),

        "formal_TFs_n": 5,

        "role": "Secondary geometric context"
    },
])

cross_tf.to_csv(
    RESULTS /
    "Step34_FINAL_CrossTF_Convergence.csv",
    index=False
)

print()
print("Cross-TF convergence:")
print(
    cross_tf.to_string(
        index=False
    )
)

print()
print("=" * 90)
print("Cell 22 complete")
print("=" * 90)

In [ ]:
# Cell 23 — Main Figure 1: primary state-displacement heatmap

print("=" * 90)
print("Cell 23 — Step34 primary heatmap")
print("=" * 90)

import matplotlib.pyplot as plt

# Median across the three primary patients
heat_rows = []

for tf in TF_ORDER:

    sub = patient_primary.loc[
        patient_primary["TF"] == tf
    ]

    heat_rows.append({
        "TF": tf,

        "Away from MS3":
            float(
                np.median(
                    sub[
                        "MS3_distance_change_excess"
                    ]
                )
            ),

        "Toward MS5":
            float(
                np.median(
                    sub[
                        "MS5_projection_excess"
                    ]
                )
            ),

        # invert sign so positive means favorable direction
        "Closer to MS5":
            float(
                -np.median(
                    sub[
                        "MS5_distance_change_excess"
                    ]
                )
            ),
    })

heat_df = (
    pd.DataFrame(heat_rows)
    .set_index("TF")
    .reindex(TF_ORDER)
)

heat_df.to_csv(
    RESULTS /
    "Step34_FINAL_PrimaryHeatmap_Data.csv"
)

values = heat_df.values

lim = float(
    np.max(
        np.abs(values)
    )
)

fig, ax = plt.subplots(
    figsize=(7.5, 5.2)
)

im = ax.imshow(
    values,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
    aspect="auto"
)

ax.set_yticks(
    np.arange(
        len(TF_ORDER)
    )
)

ax.set_yticklabels(
    TF_ORDER
)

ax.set_xticks(
    np.arange(3)
)

ax.set_xticklabels(
    [
        "Away from MS3",
        "Toward MS5",
        "Closer to MS5",
    ],
    rotation=20,
    ha="right"
)

ax.set_ylabel(
    "Virtual regulator suppression"
)

ax.set_xlabel(
    "Counterfactual state-space displacement"
)

ax.set_title(
    "Convergent displacement away from MS3 toward the MS5-like region"
)

for i, tf in enumerate(TF_ORDER):

    for j in range(3):

        ax.text(
            j,
            i,
            f"{values[i, j]:.3f}\n3/3",
            ha="center",
            va="center",
            fontsize=8
        )

cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Median excess displacement vs randomized control"
)

fig.tight_layout()

FIG1_PNG = (
    FIGURES /
    "Step34_FINAL_01_FiveTF_PrimaryStateDisplacement.png"
)

FIG1_PDF = (
    FIGURES /
    "Step34_FINAL_01_FiveTF_PrimaryStateDisplacement.pdf"
)

fig.savefig(
    FIG1_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG1_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(FIG1_PNG)
print(FIG1_PDF)

In [ ]:
# Cell 24 — Main Figure 2: patient-level displacement

print("=" * 90)
print("Cell 24 — Patient-level replication figure")
print("=" * 90)

PATIENT_MARKERS = {
    "HCC08": "o",
    "HCC09": "s",
    "HCC10": "^",
}

# Use toward-MS5 as the clearest directional endpoint
fig, ax = plt.subplots(
    figsize=(7.5, 5.5)
)

for patient in PRIMARY_PATIENTS:

    sub = (
        patient_primary.loc[
            patient_primary[
                "patient"
            ] == patient
        ]
        .set_index("TF")
        .reindex(TF_ORDER)
    )

    ax.scatter(
        np.arange(
            len(TF_ORDER)
        ),
        sub[
            "MS5_projection_excess"
        ],
        marker=PATIENT_MARKERS[
            patient
        ],
        s=55,
        label=patient
    )

ax.axhline(
    0,
    linewidth=1
)

ax.set_xticks(
    np.arange(
        len(TF_ORDER)
    )
)

ax.set_xticklabels(
    TF_ORDER
)

ax.set_ylabel(
    "Excess projection toward MS5\n(real − randomized control)"
)

ax.set_xlabel(
    "Virtual regulator suppression"
)

ax.set_title(
    "Cross-patient replication of MS5-directed displacement"
)

ax.legend(
    title="Patient",
    frameon=False
)

fig.tight_layout()

FIG2_PNG = (
    FIGURES /
    "Step34_FINAL_02_MS5Directed_PatientReplication.png"
)

FIG2_PDF = (
    FIGURES /
    "Step34_FINAL_02_MS5Directed_PatientReplication.pdf"
)

fig.savefig(
    FIG2_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    FIG2_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(FIG2_PNG)
print(FIG2_PDF)

In [ ]:
# Cell 25 — Supplementary geometric context

print("=" * 90)
print("Cell 25 — Supplementary MS5/MS6 geometric context")
print("=" * 90)

secondary_rows = []

for tf in TF_ORDER:

    sub = patient_primary.loc[
        patient_primary["TF"] == tf
    ]

    secondary_rows.append({
        "TF": tf,

        "Toward MS5":
            np.median(
                sub[
                    "MS5_projection_excess"
                ]
            ),

        "Toward MS6":
            np.median(
                sub[
                    "MS6_projection_excess"
                ]
            ),

        "Closer MS5":
            -np.median(
                sub[
                    "MS5_distance_change_excess"
                ]
            ),

        "Closer MS6":
            -np.median(
                sub[
                    "MS6_distance_change_excess"
                ]
            ),
    })

secondary_df = (
    pd.DataFrame(
        secondary_rows
    )
    .set_index("TF")
    .reindex(TF_ORDER)
)

secondary_df.to_csv(
    RESULTS /
    "Step34_FINAL_MS5_MS6_GeometricContext.csv"
)

values = secondary_df.values

lim = float(
    np.max(
        np.abs(values)
    )
)

fig, ax = plt.subplots(
    figsize=(8.5, 5.4)
)

im = ax.imshow(
    values,
    cmap="RdBu_r",
    vmin=-lim,
    vmax=lim,
    aspect="auto"
)

ax.set_yticks(
    np.arange(
        len(TF_ORDER)
    )
)

ax.set_yticklabels(
    TF_ORDER
)

ax.set_xticks(
    np.arange(
        secondary_df.shape[1]
    )
)

ax.set_xticklabels(
    secondary_df.columns,
    rotation=25,
    ha="right"
)

ax.set_title(
    "MS5- and MS6-directed geometric responses"
)

for i in range(
    secondary_df.shape[0]
):

    for j in range(
        secondary_df.shape[1]
    ):

        ax.text(
            j,
            i,
            f"{values[i,j]:.3f}",
            ha="center",
            va="center",
            fontsize=8
        )

cbar = fig.colorbar(
    im,
    ax=ax
)

cbar.set_label(
    "Median excess displacement vs randomized control"
)

fig.tight_layout()

SUPP_PNG = (
    FIGURES /
    "Step34_SUPP_01_MS5_MS6_GeometricContext.png"
)

SUPP_PDF = (
    FIGURES /
    "Step34_SUPP_01_MS5_MS6_GeometricContext.pdf"
)

fig.savefig(
    SUPP_PNG,
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    SUPP_PDF,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)

print(SUPP_PNG)
print(SUPP_PDF)

In [ ]:
# Cell 26 — Step34 FINAL LOCK

print("=" * 90)
print("Cell 26 — STEP34 FINAL LOCK")
print("=" * 90)

# ============================================================
# Formal assertions
# ============================================================

assert (
    direction_final[
        "away_from_MS3"
    ] == "3/3"
).all()

assert (
    direction_final[
        "toward_MS5"
    ] == "3/3"
).all()

assert (
    direction_final[
        "closer_to_MS5"
    ] == "3/3"
).all()


STEP34_FINAL_LOCK = {
    "status":
        "STEP34_FROZEN",

    "analysis":
        "Virtual State Transition",

    "formal_TFs":
        TF_ORDER,

    "source_state":
        "MS3",

    "primary_replication_patients":
        PRIMARY_PATIENTS,

    "formal_method":
        (
            "Native CellOracle correlation-derived transition "
            "and MS3-only row-wise embedding-shift analysis "
            "within the frozen 30-dimensional patient-only "
            "Harmony state space."
        ),

    "failed_method_audit": {
        "expression_to_Harmony_surrogate":
            "REJECTED",

        "grouped_CV_mean_R2":
            -159.6503,

        "formal_inference_from_surrogate":
            False,
    },

    "frozen_geometry": {
        "distance_MS3_MS5":
            dist_35,

        "distance_MS3_MS6":
            dist_36,

        "distance_MS5_MS6":
            dist_56,

        "cosine_MS3toMS5_vs_MS3toMS6":
            cos_35_36,

        "interpretation":
            (
                "MS5- and MS6-directed displacement are strongly "
                "aligned within the frozen state space."
            ),
    },

    "primary_endpoints": [
        "Away from MS3",
        "Projection toward MS5",
        "Reduced distance to MS5",
    ],

    "secondary_endpoints": [
        "Projection toward MS6",
        "Reduced distance to MS6",
        "First-hop transition probability",
    ],

    "primary_convergence": {
        "TFs_away_from_MS3_3of3":
            5,

        "TFs_toward_MS5_3of3":
            5,

        "TFs_closer_to_MS5_3of3":
            5,
    },

    "secondary_convergence": {
        "TFs_toward_MS6_3of3":
            4,

        "TFs_closer_to_MS6_3of3":
            3,
    },

    "interpretation_lock":
        (
            "Virtual suppression of all five frozen MS3-associated "
            "regulators reproducibly predicted counterfactual "
            "displacement away from the MS3 state and toward the "
            "frozen MS5 intermediate-state region across HCC08, "
            "HCC09, and HCC10. MS6-directed displacement was less "
            "uniform and is retained as secondary geometric context. "
            "Together with Step33, these results indicate that "
            "attenuation of the MS3 proliferative/mitotic architecture "
            "is accompanied by state-space movement toward an "
            "intermediate differentiated-state region, without "
            "establishing complete restoration of an MS6-like "
            "transcriptional identity."
        ),

    "claim_boundary":
        (
            "Counterfactual state-space displacement within the "
            "frozen CellOracle/Harmony geometry only. These results "
            "do not establish lineage conversion, causal cell-state "
            "transition, therapeutic efficacy, or experimentally "
            "validated reprogramming."
        ),

    "next_step":
        "Step35 — Virtual Drug Reversal",
}

FINAL_LOCK_PATH = (
    LOGS /
    "Step34_FINAL_LOCK.json"
)

with open(
    FINAL_LOCK_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        STEP34_FINAL_LOCK,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# Article-centered summary
# ============================================================

summary_df = pd.DataFrame([
    {
        "Step":
            "Step34",

        "Analysis":
            "Virtual State Transition",

        "Formal_TFs":
            ";".join(TF_ORDER),

        "Primary_result":
            (
                "All five perturbations showed 3/3 patient-consistent "
                "displacement away from MS3 and toward/closer to MS5."
            ),

        "Secondary_result":
            (
                "MS6-directed displacement was less uniformly "
                "reproduced across regulators."
            ),

        "Axis_geometry":
            f"MS3→MS5 vs MS3→MS6 cosine={cos_35_36:.4f}",

        "Claim_boundary":
            (
                "Counterfactual state-space displacement only; "
                "not lineage conversion."
            ),

        "Status":
            "FROZEN",
    }
])

summary_df.to_csv(
    RESULTS /
    "Step34_FINAL_ArticleCenteredSummary.csv",
    index=False
)


print("Formal TFs:", len(TF_ORDER))
print(
    "All TFs away from MS3 = 3/3:",
    True
)
print(
    "All TFs toward MS5 = 3/3:",
    True
)
print(
    "All TFs closer to MS5 = 3/3:",
    True
)

print(
    "TFs toward MS6 = 3/3:",
    "4/5"
)

print(
    "TFs closer to MS6 = 3/3:",
    "3/5"
)

print(
    "MS3→MS5 vs MS3→MS6 cosine:",
    round(cos_35_36, 4)
)

print()
print("Final lock:")
print(FINAL_LOCK_PATH)

print()
print("=" * 90)
print("STEP34 STATUS: FROZEN")
print("NEXT: STEP35 — VIRTUAL DRUG REVERSAL")
print("=" * 90)